[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nammaschool/interactive-learning/blob/main/notebooks/vectors/v07_lines.ipynb)

# Lines in 3D: aim two tunnels at each other

On 1 December 1990, two crews met under the English Channel after about 38 km of tunnelling from opposite coasts,
off by 35.8 cm sideways and 5.8 cm vertically. Two straight lines in space can be aimed at each other and still miss,
and the cross and dot products together tell you by how much.

In this notebook you can stretch an arrow into a line, watch two model tunnels pass in the dark with the shortest rope
between them, aim them yourself, and find the distance between any two lines of your own.

**How to use this notebook.** The grey boxes are cells. Click the first one and press **Shift + Enter** (or the ▶ Run
button) to run it. Run each cell in turn, from the top down. When a slider appears, drag it and watch. Change any
number and run the cell again: nothing can break. It works in your browser (no install) and in Google Colab.

In [ ]:
# Setup: in the browser version (JupyterLite) the sliders need one install; elsewhere this does nothing.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install(["ipywidgets"])

In [ ]:
# The look: one style for every figure in this series of notebooks. Run this cell first.
import math
from fractions import Fraction
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, Text, Layout

SLATE, SLATE2 = "#1f2b28", "#2a3a36"        # the slate ground
CHALK, DIM = "#ece6d6", "#93a097"           # chalk, and chalk smudge
CORAL = "#e3896b"                           # the one key thing in each figure
TEAL, SAGE, LILAC = "#6fb6bb", "#a0c27c", "#c9a0d3"   # along (dot) · the result · across (cross, the shortest rope)

plt.rcParams.update({
    "figure.facecolor": SLATE, "axes.facecolor": SLATE, "savefig.facecolor": SLATE,
    "text.color": CHALK, "axes.labelcolor": CHALK, "axes.edgecolor": DIM, "axes.titlecolor": CHALK,
    "xtick.color": DIM, "ytick.color": DIM, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#3b4c47", "grid.linewidth": 0.8,
    "font.size": 13, "axes.titlesize": 15, "axes.labelsize": 13, "legend.fontsize": 12,
    "legend.facecolor": SLATE2, "legend.edgecolor": DIM, "legend.labelcolor": CHALK, "legend.framealpha": 0.9,
    "lines.linewidth": 2.4, "figure.dpi": 100,
})

def dot(a, b): return sum(x * y for x, y in zip(a, b))
def cross(a, b): return (a[1] * b[2] - a[2] * b[1], a[2] * b[0] - a[0] * b[2], a[0] * b[1] - a[1] * b[0])
def length(a): return math.sqrt(dot(a, a))
def sub(a, b): return tuple(x - y for x, y in zip(a, b))
def at(a, b, t): return tuple(x + t * y for x, y in zip(a, b))

def nice(v):
    """A number as the board shows it: 3, −1, 0.32."""
    s = f"{v:.2f}".rstrip("0").rstrip(".") if abs(v - round(v)) > 1e-9 else str(int(round(v)))
    return s.replace("-", "−")

def column(fig, x, y, vals, color=CHALK, size=15, label=""):
    """An arrow's column of numbers, in a bracket box, at figure position (x, y)."""
    fig.text(x, y, "\n".join(nice(v) for v in vals), color=color, fontsize=size, va="center", ha="center",
             family="serif", linespacing=1.45, bbox=dict(boxstyle="square,pad=0.4", fc=SLATE, ec=DIM, lw=1.2))
    if label: fig.text(x, y + 0.13, label, color=DIM, fontsize=12, ha="center", family="serif")
    fig.text(x + 0.06, y, ".", color=SLATE, fontsize=size)          # keeps the box's edge inside a cropped figure

def space(ax, elev, azim):
    """A 3D stage on the slate: dim panes, chalk labels, the view the sliders choose."""
    ax.set_facecolor(SLATE)
    for axis in (ax.xaxis, ax.yaxis, ax.zaxis):
        axis.set_pane_color((0.16, 0.22, 0.2, 0.6)); axis.label.set_color(DIM)
    ax.view_init(elev=elev, azim=azim)
    ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_zlabel("z")

def closest(a1, b1, a2, b2):
    """The closest points of two lines that aren't parallel: the rope between them is square to both."""
    r = sub(a2, a1)
    A, B, C = dot(b1, b1), dot(b1, b2), dot(b2, b2)
    det = A * C - B * B
    lam = (dot(r, b1) * C - dot(r, b2) * B) / det
    mu = (dot(r, b1) * B - dot(r, b2) * A) / det
    return at(a1, b1, lam), at(a2, b2, mu)

## 1 · A line is a starting point and an arrow you can stretch

r = a + λb. Start at a = (2, 1, 0) and take λ steps of the arrow b. Drag λ to walk forwards and back, and drag the
arrow's numbers to point it somewhere else. The arrow and its column always say the same thing. The two view sliders
turn the picture.

In [ ]:
def stretch(lam=1.0, b1=1.0, b2=2.0, b3=2.0, turn=-60, tilt=20):
    a, b = (2, 1, 0), (b1, b2, b3)
    p = at(a, b, lam)
    fig = plt.figure(figsize=(9, 5.6))
    ax = fig.add_axes([0.0, 0.0, 0.68, 1.0], projection="3d")
    space(ax, tilt, turn)
    ts = np.linspace(-3, 3, 2)
    ax.plot([a[0] + t * b[0] for t in ts], [a[1] + t * b[1] for t in ts], [a[2] + t * b[2] for t in ts],
            color=DIM, lw=1.4, ls="--")
    ax.quiver(*a, *b, color=CHALK, lw=2.4, arrow_length_ratio=0.15)
    ax.scatter(*a, color=CHALK, s=40)
    ax.scatter(*p, color=CORAL, s=70, depthshade=False)
    ax.set_xlim(-4, 8); ax.set_ylim(-5, 7); ax.set_zlim(-6, 6)
    ax.set_title(f"λ = {nice(lam)}:  r = ({nice(p[0])}, {nice(p[1])}, {nice(p[2])})", color=CHALK)
    column(fig, 0.74, 0.55, b, label="b")
    column(fig, 0.84, 0.55, p, color=CORAL, label="r")
    plt.show()

interact(stretch, lam=FloatSlider(value=1, min=-2, max=2, step=0.5, description="λ (steps)"),
         b1=FloatSlider(value=1, min=-3, max=3, step=0.5, description="b: x"),
         b2=FloatSlider(value=2, min=-3, max=3, step=0.5, description="b: y"),
         b3=FloatSlider(value=2, min=-3, max=3, step=0.5, description="b: z"),
         turn=FloatSlider(value=-60, min=-180, max=180, step=10, description="view: turn"),
         tilt=FloatSlider(value=20, min=-10, max=80, step=5, description="view: tilt"));

## 2 · Two tunnels in a model mountain

A model in metres, not the real survey. The west mouth is at (0, 0, 0) and the east mouth at (1000, 0, −3). The west
crew heads along (100, 1, 0) and the east crew along (−100, 2, 1). They aren't parallel and they never meet: they're
skew lines. The coral rope is the shortest one between them, square to both, along b₁ × b₂.
The sideways and up-down scales are stretched so you can see the gap (the true picture is a kilometre long and a
few metres across).

In [ ]:
W, E = (0, 0, 0), (1000, 0, -3)
B1, B2 = (100, 1, 0), (-100, 2, 1)

def tunnels(east_z=-3.0, turn=-70, tilt=18):
    E = (1000, 0, east_z)
    N = cross(B1, B2)
    gap = abs(dot(sub(E, W), N)) / length(N)
    PW, PE = closest(W, B1, E, B2)
    fig = plt.figure(figsize=(11, 5.2))
    ax = fig.add_subplot(1, 2, 1, projection="3d")
    space(ax, tilt, turn)
    ax.plot([0, 1000], [0, 10], [0, 0], color=CHALK, label="west heading")
    ax.plot([1000, 0], [0, 20], [east_z, east_z + 10], color=TEAL, label="east heading")
    ax.plot(*zip(PW, PE), color=CORAL, lw=4)
    ax.scatter(*W, color=CHALK, s=30); ax.scatter(*E, color=TEAL, s=30)
    ax.set_title("the two headings, end to end", fontsize=13)
    ax.legend(loc="upper left", fontsize=10)
    zx = fig.add_subplot(1, 2, 2, projection="3d")
    space(zx, tilt, turn)
    s = np.linspace(-0.4, 0.4, 2)
    lam0, mu0 = PW[0] / 100, (1000 - PE[0]) / 100
    zx.plot([100 * (lam0 + k) for k in s], [lam0 + k for k in s], [0, 0], color=CHALK)
    zx.plot([1000 - 100 * (mu0 + k) for k in s], [2 * (mu0 + k) for k in s],
            [east_z + mu0 + k for k in s], color=TEAL)
    zx.plot(*zip(PW, PE), color=CORAL, lw=4)
    zx.set_title(f"close-up: the tunnels pass {100 * gap:.1f} cm apart", fontsize=13, color=CORAL)
    plt.show()
    return gap

gap = tunnels()
print(f"shortest rope: |(1000, 0, −3) · (1, −100, 300)| ÷ √100001 = 100 ÷ {length(cross(B1, B2)):.1f} = {gap:.2f} m")

## 3 · Aim the tunnels: drag the height of the east mouth

Move the east mouth up or down and watch the shortest rope shrink and grow. At one height it disappears, because
the tunnels meet. Which height? (The rope from mouth to mouth is (1000, 0, z): when is its dot product with
(1, −100, 300) zero?)

In [ ]:
def aim(east_mouth_height=-3.0):
    g = tunnels(east_mouth_height)
    if g < 0.005: print("the rope has disappeared: the two tunnels meet")
    else: print(f"the tunnels miss by {100 * g:.1f} cm")

interact(aim, east_mouth_height=FloatSlider(value=-3.0, min=-4.0, max=-2.0, step=1 / 30, readout_format=".2f",
                                            description="east mouth z", layout=Layout(width="520px")));

## 4 · Your own lines

Type each line as its point and its direction: `1 0 2 ; 2 1 -2` means r = (1, 0, 2) + λ(2, 1, −2). Press Enter, and
the cell tells you whether the lines are parallel, meet or are skew, and how far apart they are. It starts with
the two model tunnels; try the exam question from the video: `1 0 2 ; 2 1 -2` and `0 1 1 ; 1 1 1`.

In [ ]:
def numbers(s):
    s = s.replace("−", "-").replace(",", " ").replace("(", " ").replace(")", " ").replace(";", " ")
    return [float(v) for v in s.split()]

def distance(a1, b1, a2, b2):
    """The shortest distance between two lines: the cross-and-dot formula, or the parallel one when b₁ × b₂ = 0."""
    n = cross(b1, b2); r = sub(a2, a1)
    if length(n) < 1e-12:
        return "parallel", length(cross(b1, r)) / length(b1)
    d = abs(dot(r, n)) / length(n)
    return ("they meet" if d < 1e-9 else "skew"), d

def your_lines(line_1="0 0 0 ; 100 1 0", line_2="1000 0 -3 ; -100 2 1"):
    try:
        p, q = numbers(line_1), numbers(line_2)
        a1, b1, a2, b2 = tuple(p[:3]), tuple(p[3:6]), tuple(q[:3]), tuple(q[3:6])
        assert len(p) == 6 and len(q) == 6 and length(b1) > 0 and length(b2) > 0
    except Exception:
        print("Type six numbers for each line: the point, then the direction."); return
    kind, d = distance(a1, b1, a2, b2)
    print(f"line 1: r = {tuple(map(nice, a1))} + λ{tuple(map(nice, b1))}".replace("'", ""))
    print(f"line 2: r = {tuple(map(nice, a2))} + μ{tuple(map(nice, b2))}".replace("'", ""))
    print(f"{kind}: shortest distance {d:.4f}")

In [ ]:
interact(your_lines, line_1=Text(value="0 0 0 ; 100 1 0", continuous_update=False, description="line 1",
                                 layout=Layout(width="420px")),
         line_2=Text(value="1000 0 -3 ; -100 2 1", continuous_update=False, description="line 2", layout=Layout(width="420px")));

**Try it.** The challenge from the video: in the model, how far below the west mouth should the east mouth have been,
for the two headings to meet exactly? Work it out on paper (make the dot product zero), then check it with the slider
in section 3.

## Sources
The Channel Tunnel breakthrough: "Tunnels – Topography", *Proceedings of the ICE: Civil Engineering Supplement* 102
(1994), pp. 58–62 (TRID 413602). NCERT Mathematics Class XII, ch. 11 (Three Dimensional Geometry); CBSE Mathematics
syllabus 2026–27. All retrieved 2026-10-26. The tunnel numbers are our own model, not the real survey.